In [3]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"
DATASET = "production"
URL = f"https://{PROJECT}.api.sanity.io/v2021-06-07/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

# Pobierz 20 produktów z różnych kategorii z pełnym łańcuchem kategorii (do 4 poziomów)
QUERY = """
*[_type == "product" && defined(category) && defined(slug)] | order(name asc) [0...200] {
  "sku": sku,
  "slug": slug.current,
  "name": name,
  "cat1": category->name,
  "cat1slug": category->slug.current,
  "cat2": category->parent->name,
  "cat2slug": category->parent->slug.current,
  "cat3": category->parent->parent->name,
  "cat3slug": category->parent->parent->slug.current,
  "cat4": category->parent->parent->parent->name,
  "cat4slug": category->parent->parent->parent->slug.current
}
"""

resp = requests.get(URL, headers=HEADERS, params={"query": QUERY})
data = resp.json()["result"]
print(f"Pobrano: {len(data)} produktów")

# Pokaż próbkę z różnych kategorii top-level - zbierz po 3 z każdej
seen_top = {}
sample = []
for p in data:
    top = p["cat4slug"] or p["cat3slug"] or p["cat2slug"] or p["cat1slug"]
    if top and top not in seen_top:
        seen_top[top] = 0
    if top and seen_top[top] < 3:
        sample.append(p)
        seen_top[top] += 1

print(f"\nPróbka: {len(sample)} produktów z {len(seen_top)} kategorii top-level")
for p in sample:
    # Zbuduj ścieżkę od góry do dołu
    path_parts = []
    for name, slug in [(p['cat4'], p['cat4slug']), (p['cat3'], p['cat3slug']), (p['cat2'], p['cat2slug']), (p['cat1'], p['cat1slug'])]:
        if name:
            path_parts.append(f"{name}")
    path = " > ".join(path_parts)
    print(f"  SKU={p['sku']} | {p['name'][:45]}")
    print(f"    mediabud: Strona główna > {path}")
    print()


Pobrano: 200 produktów

Próbka: 21 produktów z 7 kategorii top-level
  SKU=ATL-WODER-B-25 | A Uszczelniajaca Atlas Woder Sx Do Izolacji F
    mediabud: Strona główna > Chemia budowlana > Zaprawy > Zaprawy uszczelniające

  SKU=ATL-WODER-D-5 | A Uszczelniajaca Atlas Woder Sx Do Izolacji F
    mediabud: Strona główna > Chemia budowlana > Zaprawy > Zaprawy uszczelniające

  SKU=P-0298405 | Aceton Cazet Kampinos 0,5 l
    mediabud: Strona główna > Farby i rozpuszczalniki > Rozpuszczalniki

  SKU=P-0298406 | Aceton Cazet Kampinos 5 l
    mediabud: Strona główna > Farby i rozpuszczalniki > Rozpuszczalniki

  SKU=P-0196018 | Aceton techniczny Dorex 0,5 l
    mediabud: Strona główna > Farby i rozpuszczalniki > Rozpuszczalniki

  SKU=P-0093914 | Adapter JAWAR K 080 0,15 m
    mediabud: Strona główna > Stropy i ściany > Systemy kominowe > Akcesoria do kominów

  SKU=P-0232201 | Adapter Schiedel TER 100/-/0,4/316/BA/UK+
    mediabud: Strona główna > Stropy i ściany > Systemy kominowe > Kominy sta

In [7]:

import requests, json, time
from bs4 import BeautifulSoup

HEADERS_WEB = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"}

# Wybrane SKU do weryfikacji (P-XXXXXXX z Sanity)
TEST_SKUS = [
    ("P-0298405", "Aceton Cazet Kampinos 0,5 l"),
    ("P-0093914", "Adapter JAWAR K 080 0,15 m"),
    ("P-0105150", "Akcesoria KCS narożnik"),
    ("P-0130278", "Bariera przeciwśniegowa aluminiowa MDM"),
    ("P-0094596", "Bariera przeciwśniegowa stalowa MDM"),
    ("P-0044410", "Blachowkręty Siniat Cementex"),
    ("P-0310126", "Akcelerator Remmers ACC H"),
    ("P-0301807", "Adapter do szpachli wykończeniowej"),
    ("P-0299904", "Beczka z tworzywa OFO"),
    ("P-0230752", "Belki montażowe Velux"),
]

def get_bechcicki_breadcrumb(sku):
    """Pobierz breadcrumb z bechcicki.pl dla danego SKU"""
    # Format URL: https://www.bechcicki.pl/p/P-XXXXXXX
    url = f"https://www.bechcicki.pl/p/{sku}"
    try:
        r = requests.get(url, headers=HEADERS_WEB, timeout=15, allow_redirects=True)
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}", r.url
        
        soup = BeautifulSoup(r.text, "html.parser")
        
        # Szukaj JSON-LD BreadcrumbList
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                ld = json.loads(script.string)
                items = []
                if isinstance(ld, list):
                    for item in ld:
                        if item.get("@type") == "BreadcrumbList":
                            items = [el["name"] for el in item.get("itemListElement", [])]
                elif ld.get("@type") == "BreadcrumbList":
                    items = [el["name"] for el in ld.get("itemListElement", [])]
                if items:
                    return items, "ok", r.url
            except:
                pass
        
        # Fallback: breadcrumb HTML
        bc = soup.find("nav", {"aria-label": "breadcrumb"}) or soup.find(class_=lambda c: c and "breadcrumb" in c.lower())
        if bc:
            parts = [a.get_text(strip=True) for a in bc.find_all("a")]
            return parts, "html-fallback", r.url
        
        return None, "no-breadcrumb", r.url
    except Exception as e:
        return None, str(e)[:60], ""

results = []
for sku, name in TEST_SKUS:
    bc, status, final_url = get_bechcicki_breadcrumb(sku)
    results.append({"sku": sku, "name": name, "breadcrumb": bc, "status": status, "url": final_url})
    print(f"{sku}: {status} → {' > '.join(bc) if bc else 'BRAK'}")
    time.sleep(0.5)

print(f"\nZebrано {len(results)} wyników")


P-0298405: HTTP 404 → BRAK


P-0093914: HTTP 404 → BRAK


P-0105150: HTTP 404 → BRAK


P-0130278: HTTP 404 → BRAK


P-0094596: HTTP 404 → BRAK


...

P-0310126: HTTP 404 → BRAK


P-0301807: HTTP 404 → BRAK


P-0299904: HTTP 404 → BRAK


P-0230752: HTTP 404 → BRAK



Zebrано 10 wyników


In [11]:

import requests, json, time
from bs4 import BeautifulSoup

HEADERS_WEB = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"}

def get_bechcicki_breadcrumb(sku):
    pid = sku  # np. "P-0298405"
    url = f"https://www.bechcicki.pl/{pid}-id-p-{pid}"
    try:
        r = requests.get(url, headers=HEADERS_WEB, timeout=15, allow_redirects=True)
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}", url
        soup = BeautifulSoup(r.text, "html.parser")
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                ld = json.loads(script.string or "")
                if isinstance(ld, list):
                    for item in ld:
                        if item.get("@type") == "BreadcrumbList":
                            items = [el["name"] for el in item.get("itemListElement", [])]
                            return items, "ok", r.url
                elif ld.get("@type") == "BreadcrumbList":
                    items = [el["name"] for el in ld.get("itemListElement", [])]
                    return items, "ok", r.url
            except:
                pass
        # Fallback HTML
        bc = soup.find(class_=lambda c: c and "breadcrumb" in c.lower())
        if bc:
            parts = [el.get_text(strip=True) for el in bc.find_all(["a","span","li"]) if el.get_text(strip=True)]
            return parts, "html", r.url
        return None, "no-bc", r.url
    except Exception as e:
        return None, str(e)[:60], url

# Produkty testowe z próbki Sanity (z różnych kategorii top-level)
TEST = [
    ("P-0298405", "Aceton Cazet Kampinos 0,5 l",     "Farby i rozpuszczalniki > Rozpuszczalniki"),
    ("P-0093914", "Adapter JAWAR K 080 0,15 m",       "Stropy i ściany > Systemy kominowe > Akcesoria do kominów"),
    ("P-0105150", "Akcesoria KCS narożnik",            "Izolacje > Akcesoria do izolacji > Profile i narożniki"),
    ("P-0130278", "Bariera przec. śniegowa alum. MDM", "Dachy > Zabezpieczenia przeciwśniegowe"),
    ("P-0044410", "Blachowkręty Siniat Cementex",      "Sucha zabudowa > Mocowania > Wkręty"),
    ("P-0301807", "Adapter do szpachli Blue Dolphin",  "Narzędzia i mocowania > Akcesoria malarskie"),
    ("P-0299904", "Beczka z tworzywa OFO 200 l",       "Narzędzia i mocowania > Akcesoria malarskie > Wiadra"),
    ("P-0230752", "Belki montażowe Velux DUO EMT",     "Dachy > Okna dachowe > Kołnierze"),
    ("P-0310126", "Akcelerator Remmers ACC H",          "Chemia budowlana > Spoiny"),
    ("P-0367914", "Blachowkręty Siniat Nida 3,5×25 taśma", "Sucha zabudowa > Mocowania > Wkręty"),
]

results = []
for sku, name, mediabud_path in TEST:
    bc, status, final_url = get_bechcicki_breadcrumb(sku)
    results.append({
        "sku": sku,
        "name": name,
        "mediabud": mediabud_path,
        "bechcicki": " > ".join(bc) if bc else "BRAK",
        "status": status,
    })
    print(f"[{status}] {sku}")
    print(f"  mediabud:  Strona główna > {mediabud_path}")
    print(f"  bechcicki: {' > '.join(bc) if bc else 'BRAK'}")
    print()
    time.sleep(0.8)


[HTTP 500] P-0298405
  mediabud:  Strona główna > Farby i rozpuszczalniki > Rozpuszczalniki
  bechcicki: BRAK



[HTTP 500] P-0093914
  mediabud:  Strona główna > Stropy i ściany > Systemy kominowe > Akcesoria do kominów
  bechcicki: BRAK



[HTTP 500] P-0105150
  mediabud:  Strona główna > Izolacje > Akcesoria do izolacji > Profile i narożniki
  bechcicki: BRAK



[HTTP 500] P-0130278
  mediabud:  Strona główna > Dachy > Zabezpieczenia przeciwśniegowe
  bechcicki: BRAK



[HTTP 500] P-0044410
  mediabud:  Strona główna > Sucha zabudowa > Mocowania > Wkręty
  bechcicki: BRAK



[HTTP 500] P-0301807
  mediabud:  Strona główna > Narzędzia i mocowania > Akcesoria malarskie
  bechcicki: BRAK



[HTTP 500] P-0299904
  mediabud:  Strona główna > Narzędzia i mocowania > Akcesoria malarskie > Wiadra
  bechcicki: BRAK



[HTTP 500] P-0230752
  mediabud:  Strona główna > Dachy > Okna dachowe > Kołnierze
  bechcicki: BRAK



[HTTP 500] P-0310126
  mediabud:  Strona główna > Chemia budowlana > Spoiny
  bechcicki: BRAK



[HTTP 500] P-0367914
  mediabud:  Strona główna > Sucha zabudowa > Mocowania > Wkręty
  bechcicki: BRAK



In [15]:

import json, re

JSONL = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/reimport_scraped.jsonl"

# Załaduj wszystkie produkty z JSONL — zbieramy pole "breadcrumb" lub "categories"
bech_map = {}   # sku -> dane
with open(JSONL) as f:
    for line in f:
        try:
            d = json.loads(line)
            sku = d.get("sku","")
            if sku:
                bech_map[sku] = d
        except:
            pass

print(f"Załadowano {len(bech_map)} produktów z JSONL")

# Sprawdź jakie pola są dostępne w pierwszym produkcie
first = next(iter(bech_map.values()))
print(f"\nDostępne pola: {list(first.keys())}")
print(f"\nPrzykład: {first}")


Załadowano 15654 produktów z JSONL

Dostępne pola: ['sku', 'name', 'categoryPath', 'brand', 'technicalSpec', 'description', 'ean', 'imageUrl', 'status']

Przykład: {'sku': 'P-0000049', 'name': 'Gładź cementowo-polimerowa Cekol C-35 biała 5 kg', 'categoryPath': ['Chemia budowlana', 'Gipsy i gładzie', 'Gładzie gipsowe w proszku'], 'brand': 'Cekol', 'technicalSpec': [{'label': 'Ilość na palecie', 'value': '48'}, {'label': 'Rodzaj gładzi/gipsu', 'value': 'Gładź cementowa'}, {'label': 'Metoda aplikacji', 'value': 'ręczna'}, {'label': 'Metoda aplikacji', 'value': 'maszynowo'}, {'label': 'Temperatura pracy ', 'value': 'od 5°C do 25°C'}, {'label': 'Zastosowanie', 'value': 'wewnętrzne'}, {'label': 'Zastosowanie', 'value': 'zewnętrzne'}, {'label': 'Klasa reakcji na ogień', 'value': 'A1'}, {'label': 'Rodzaj pomieszczenia', 'value': 'na zewnątrz budynku'}, {'label': 'Rodzaj pomieszczenia', 'value': 'kuchnia'}, {'label': 'Rodzaj pomieszczenia', 'value': 'łazienka'}, {'label': 'Maksymalna grubość wa

In [19]:

import json, requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
URL = f"https://{PROJECT}.api.sanity.io/v2021-06-07/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

# Pobierz z Sanity pełne łańcuchy kategorii dla WSZYSTKICH produktów (w partiach)
all_sanity = {}
offset = 0
while True:
    q = f"""*[_type == "product" && defined(category)] [{offset}...{offset+2000}] {{
      "sku": sku,
      "c1": category->name,
      "c2": category->parent->name,
      "c3": category->parent->parent->name,
      "c4": category->parent->parent->parent->name
    }}"""
    r = requests.get(URL, headers=HEADERS, params={"query": q})
    batch = r.json().get("result", [])
    if not batch:
        break
    for p in batch:
        if p.get("sku"):
            # Buduj ścieżkę od korzenia (odwróć kolejność)
            chain = [n for n in [p.get("c4"), p.get("c3"), p.get("c2"), p.get("c1")] if n]
            all_sanity[p["sku"]] = chain
    offset += 2000
    if len(batch) < 2000:
        break

print(f"Sanity: {len(all_sanity)} produktów z kategorią")

# Porównaj z bechcicki.pl (JSONL)
diffs = []     # produkty gdzie ścieżka się różni
matches = 0
no_bech = 0

for sku, sanity_path in all_sanity.items():
    bech_data = bech_map.get(sku)
    if not bech_data:
        no_bech += 1
        continue
    bech_path = bech_data.get("categoryPath", [])
    
    sanity_str = " > ".join(sanity_path)
    bech_str   = " > ".join(bech_path)
    
    if sanity_str == bech_str:
        matches += 1
    else:
        diffs.append({
            "sku": sku,
            "name": bech_data.get("name","")[:50],
            "bechcicki": bech_str,
            "mediabud":  sanity_str,
        })

print(f"Identyczne ścieżki: {matches}")
print(f"Różne ścieżki:      {len(diffs)}")
print(f"Brak w JSONL:       {no_bech}")
print(f"\nOdsetek zgodności: {matches/(matches+len(diffs))*100:.1f}%")


Sanity: 15834 produktów z kategorią
Identyczne ścieżki: 15654
Różne ścieżki:      0
Brak w JSONL:       180

Odsetek zgodności: 100.0%


In [23]:

# Pokaż 15 reprezentatywnych przykładów (różne top-level kategorie)
print("=== PRZYKŁADOWE ŚCIEŻKI KATEGORII: bechcicki.pl (JSONL) vs Sanity ===\n")

# Zbierz po 1-2 przykłady z każdej top-level kategorii
by_top = {}
for sku, sanity_path in all_sanity.items():
    bd = bech_map.get(sku)
    if not bd:
        continue
    top = sanity_path[0] if sanity_path else "?"
    if top not in by_top:
        by_top[top] = []
    if len(by_top[top]) < 2:
        by_top[top].append({
            "sku": sku,
            "name": bd.get("name","")[:55],
            "bechcicki": " > ".join(bd.get("categoryPath",[])),
            "sanity":    " > ".join(sanity_path),
        })

for top, items in sorted(by_top.items()):
    print(f"{'─'*70}")
    print(f"TOP LEVEL: {top}")
    for it in items:
        match = "✅" if it["bechcicki"] == it["sanity"] else "❌"
        print(f"  {match} SKU: {it['sku']}")
        print(f"     Nazwa:     {it['name']}")
        print(f"     bechcicki: {it['bechcicki']}")
        print(f"     Sanity:    {it['sanity']}")
    print()

print(f"{'═'*70}")
print(f"PODSUMOWANIE:")
print(f"  Produkty w Sanity z kategorią: 15 834")
print(f"  Produkty z identyczną ścieżką: 15 654 (100% tych z JSONL)")
print(f"  Produkty bez danych JSONL (nowe/bez SKU): 180")
print(f"  ✅ Dane kategorii w Sanity = 1:1 z bechcicki.pl")


=== PRZYKŁADOWE ŚCIEŻKI KATEGORII: bechcicki.pl (JSONL) vs Sanity ===

──────────────────────────────────────────────────────────────────────
TOP LEVEL: Chemia budowlana
  ✅ SKU: P-0000015
     Nazwa:     Masa asfaltowa Izolacja Jarocin Jarlep K 10 kg
     bechcicki: Chemia budowlana > Zaprawy > Zaprawy do renowacji
     Sanity:    Chemia budowlana > Zaprawy > Zaprawy do renowacji
  ✅ SKU: P-0000049
     Nazwa:     Gładź cementowo-polimerowa Cekol C-35 biała 5 kg
     bechcicki: Chemia budowlana > Gipsy i gładzie > Gładzie gipsowe w proszku
     Sanity:    Chemia budowlana > Gipsy i gładzie > Gładzie gipsowe w proszku

──────────────────────────────────────────────────────────────────────
TOP LEVEL: Dachy
  ✅ SKU: P-0000720
     Nazwa:     Okno dachowe Fakro PTP-V U4 RAL 7022 66x118 3-szybowe P
     bechcicki: Dachy > Okna dachowe i akcesoria > Okna dachowe
     Sanity:    Dachy > Okna dachowe i akcesoria > Okna dachowe
  ✅ SKU: P-0000721
     Nazwa:     Okno dachowe Velux GLL MK04 106

In [27]:

import requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
URL = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
H = {"Authorization": f"Bearer {TOKEN}"}

# Znajdź slugi dla wybranych SKU - bierzemy produkty z różnych kategorii + różną głębokością
skus = ["P-0000049", "P-0002687", "P-0000720", "P-0000281", "P-0000301", "P-0006450", "P-0005440", "P-0000179"]
q = f'*[_type=="product" && sku in {json.dumps(skus)}]{{ "sku":sku, "slug":slug.current, name }}'
r = requests.get(URL, headers=H, params={"query": q})
for p in r.json()["result"]:
    print(f'{p["sku"]:15} slug={p["slug"]:50} | {p["name"][:45]}')


P-0000049       slug=gladz-cementowo-polimerowa-cekol-c-35-biala-5-kg   | Gładź cementowo-polimerowa Cekol C-35 biała 5
P-0000179       slug=styropian-fundamentowy-arsanit-thermo-aqua-standard-eps-100-036-gr-10-cm-30-m203-m3 | Styropian fundamentowy Arsanit Thermo Aqua St
P-0000281       slug=lakier-poliuretanowo-akrylowy-do-drewna-drewnochron-parkiet-expres-polmat-5-l | Lakier poliuretanowo-akrylowy do drewna Drewn
P-0000301       slug=wkret-koelner-do-plyt-g-k-35x35-mm-do-stali-150-szt | Wkręt Koelner do płyt G-K 3,5x35 mm do stali 
P-0000720       slug=okno-dachowe-fakro-ptp-v-u4-ral-7022-66x118-3-szybowe-pvc | Okno dachowe Fakro PTP-V U4 RAL 7022 66x118 3
P-0002687       slug=bloczek-hh-gold-40-500-pwu-175250625-mm            | Bloczek H+H Gold+ 4,0-500 PWU 175/250/625 mm
P-0005440       slug=plytka-elewacyjna-paradyz-viano-245x66-beige-mat-gat1 | Płytka elewacyjna Paradyż Viano 24.5x6.6 beig
P-0006450       slug=szlifierka-katowa-modeco-expert-230-mm-2300-w      | Szlifierka kątow

In [31]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
URL = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
H = {"Authorization": f"Bearer {TOKEN}"}

# Pobierz aktualne URL zdjęć dla kilku produktów
q = '''*[_type=="product" && sku in ["P-0002687","P-0000049","P-0000720","P-0006450"]] {
  sku, name, "img": images[0].asset->url
}'''
r = requests.get(URL, headers=H, params={"query": q})
for p in r.json()["result"]:
    img = p.get("img","brak")
    print(f'{p["sku"]} | {p["name"][:45]}')
    print(f'  URL: {img}')
    print()


P-0000049 | Gładź cementowo-polimerowa Cekol C-35 biała 5
  URL: https://cdn.sanity.io/images/nzcwegq7/production/42e4bfad8144d16b1f4778fe1d1d8246cfd779fb-1200x1200.webp

P-0000720 | Okno dachowe Fakro PTP-V U4 RAL 7022 66x118 3
  URL: https://cdn.sanity.io/images/nzcwegq7/production/01635b79a655ea325b4221416a05c75f7636b3c9-1200x1200.jpg

P-0002687 | Bloczek H+H Gold+ 4,0-500 PWU 175/250/625 mm
  URL: https://cdn.sanity.io/images/nzcwegq7/production/f77b98e2ecc0a473c9b60140b20aa1b22cd75765-1200x1200.jpg

P-0006450 | Szlifierka kątowa Modeco Expert 230 mm 2300 W
  URL: https://cdn.sanity.io/images/nzcwegq7/production/ef6670c385edcc8495746502b792d13c2184e4eb-1200x1200.webp



In [35]:

import json

LOG = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/img_to_fix_log.json"
with open(LOG) as f:
    log = json.load(f)

print(f"Typ: {type(log)}")
if isinstance(log, list):
    print(f"Rekordów: {len(log)}")
    print(f"Przykład: {log[0]}")
elif isinstance(log, dict):
    print(f"Klucze: {list(log.keys())[:10]}")
    first_key = list(log.keys())[0]
    print(f"Przykład [{first_key}]: {log[first_key]}")


Typ: <class 'list'>
Rekordów: 14823
Przykład: {'_id': 'prod-styropian-swisspor-eps70', 'url': 'https://cdn.sanity.io/images/nzcwegq7/production/b4df9ba85c5a974830a2fdc2993cc69b5ee07aa7-1200x1200.jpg', 'status': 'skip_dark'}


In [39]:

import json
from collections import Counter

LOG = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/img_to_fix_log.json"
with open(LOG) as f:
    log = json.load(f)

statuses = Counter(e["status"] for e in log)
print("Statusy:", dict(statuses))

# Wyciągnij zaktualizowane produkty (mają URL w Sanity CDN)
updated = [e for e in log if e["status"] == "updated"]
print(f"\nZaktualizowane: {len(updated)}")
print(f"Przykład updated: {updated[0]}")

# Sprawdź czy mają też originalUrl
has_orig = sum(1 for e in updated if "originalUrl" in e or "bechcickiUrl" in e or "orig" in e)
print(f"Mają originalUrl: {has_orig}")
print(f"Wszystkie pola: {list(updated[0].keys())}")


Statusy: {'skip_dark': 5122, 'updated(130KB)': 85, 'updated(115KB)': 46, 'updated(179KB)': 11, 'updated(169KB)': 21, 'updated(68KB)': 63, 'updated(72KB)': 88, 'updated(118KB)': 80, 'updated(96KB)': 115, 'updated(81KB)': 43, 'updated(258KB)': 2, 'updated(141KB)': 52, 'updated(139KB)': 477, 'error_upload': 28, 'updated(140KB)': 30, 'updated(135KB)': 41, 'updated(61KB)': 55, 'updated(114KB)': 46, 'updated(125KB)': 34, 'updated(104KB)': 114, 'updated(217KB)': 4, 'updated(108KB)': 74, 'updated(52KB)': 97, 'updated(206KB)': 48, 'updated(99KB)': 216, 'updated(177KB)': 22, 'updated(109KB)': 72, 'updated(218KB)': 13, 'updated(187KB)': 17, 'updated(122KB)': 90, 'updated(266KB)': 4, 'updated(153KB)': 48, 'updated(76KB)': 96, 'updated(95KB)': 104, 'updated(175KB)': 14, 'updated(192KB)': 27, 'updated(149KB)': 25, 'updated(56KB)': 73, 'updated(69KB)': 114, 'updated(134KB)': 46, 'updated(126KB)': 70, 'updated(63KB)': 71, 'updated(263KB)': 1, 'updated(124KB)': 48, 'updated(102KB)': 38, 'updated(128KB)

IndexError: list index out of range

In [43]:

import json

LOG = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/img_to_fix_log.json"
with open(LOG) as f:
    log = json.load(f)

# Status format: "updated(XXkB)"
updated = [e for e in log if e["status"].startswith("updated(")]
print(f"Zaktualizowane: {len(updated)}")
print(f"Przykład: {updated[0]}")
print(f"Pola: {list(updated[0].keys())}")

# Zapis listy do pliku - _id + url CDN
with open("/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/updated_products.json","w") as f:
    json.dump(updated, f)
print(f"\nZapisano {len(updated)} wpisów do updated_products.json")


Zaktualizowane: 9642
Przykład: {'_id': 'product-p0000015', 'url': 'https://cdn.sanity.io/images/nzcwegq7/production/bf71ad102869390c802d48f550cdb8764e20c607-700x700.webp', 'status': 'updated(130KB)'}
Pola: ['_id', 'url', 'status']

Zapisano 9642 wpisów do updated_products.json


In [47]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
URL = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
H = {"Authorization": f"Bearer {TOKEN}"}

q = '*[_type=="product" && sku in ["P-0002687","P-0000049","P-0006450","P-0000179"]] { sku, name, "img": images[0].asset->url }'
r = requests.get(URL, headers=H, params={"query": q})
for p in r.json()["result"]:
    print(f'{p["sku"]} | {p["name"][:40]}')
    print(f'  {p.get("img","brak")}')
    print()


P-0000049 | Gładź cementowo-polimerowa Cekol C-35 bi
  https://cdn.sanity.io/images/nzcwegq7/production/42e4bfad8144d16b1f4778fe1d1d8246cfd779fb-1200x1200.webp

P-0000179 | Styropian fundamentowy Arsanit Thermo Aq
  https://cdn.sanity.io/images/nzcwegq7/production/07d338bce9aca91536d4420286c2cc13a91075a7-1200x1200.webp

P-0002687 | Bloczek H+H Gold+ 4,0-500 PWU 175/250/62
  https://cdn.sanity.io/images/nzcwegq7/production/f77b98e2ecc0a473c9b60140b20aa1b22cd75765-1200x1200.jpg

P-0006450 | Szlifierka kątowa Modeco Expert 230 mm 2
  https://cdn.sanity.io/images/nzcwegq7/production/ef6670c385edcc8495746502b792d13c2184e4eb-1200x1200.webp



In [51]:

import json

REVERT_LOG = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/revert_log.json"
with open(REVERT_LOG) as f:
    rlog = json.load(f)

from collections import Counter
statuses = Counter(e["status"] for e in rlog)
print(f"Statusy w revert_log: {dict(statuses)}\n")

# Szukaj bloczka H+H
target_id = "product-p0002687"
hit = next((e for e in rlog if e["_id"] == target_id), None)
print(f"Bloczek H+H ({target_id}): {hit}")

# Pokaż kilka "reverted"
reverted = [e for e in rlog if e["status"] == "reverted"][:3]
print(f"\nPrzykłady reverted:")
for e in reverted:
    print(f"  {e}")

# Pokaż kilka "ok" - co to za ratio?
ok_items = [e for e in rlog if e["status"] == "ok"]
ratios = [e.get("ratio",0) for e in ok_items]
import statistics
print(f"\nOK items: {len(ok_items)}, ratio avg={statistics.mean(ratios):.5f}, max={max(ratios):.5f}")

# Reverted ratio
rev_items = [e for e in rlog if e["status"] == "reverted"]
rev_ratios = [e.get("ratio",0) for e in rev_items]
print(f"Reverted items: {len(rev_items)}, ratio avg={statistics.mean(rev_ratios):.5f}, max={max(rev_ratios):.5f}")


Statusy w revert_log: {'ok': 7619, 'reverted': 2002, 'upload_failed': 15, 'no_original': 6}

Bloczek H+H (product-p0002687): {'_id': 'product-p0002687', 'status': 'ok', 'ratio': 0.0}

Przykłady reverted:
  {'_id': 'product-p0000322', 'status': 'reverted', 'ratio': 0.01355, 'asset': 'image-b65578e36ab1e0e0aae0ec51cf1cbc25898bd00b-800x800-webp'}
  {'_id': 'product-p0000218', 'status': 'reverted', 'ratio': 0.0058, 'asset': 'image-cd1148fb22d08204f046429961823593936d15a2-400x400-webp'}
  {'_id': 'product-p0000417', 'status': 'reverted', 'ratio': 0.02142, 'asset': 'image-e84cd48dcbd5f0c9d92dbf17603fbb7a4bec8437-1200x1200-webp'}

OK items: 7619, ratio avg=0.00081, max=0.00494
Reverted items: 2002, ratio avg=0.04474, max=0.42841


In [3]:

import json, requests, io
from PIL import Image
import numpy as np

WS = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

# 1. Sprawdź flood-fill log (img_to_fix_log.json)
with open(f"{WS}/img_to_fix_log.json") as f:
    fflog = json.load(f)
hit_ff = next((e for e in fflog if e["_id"] == "product-p0002687"), None)
print(f"Flood-fill log: {hit_ff}")

# 2. Sprawdź updated_products.json
with open(f"{WS}/updated_products.json") as f:
    updated = json.load(f)
hit_up = next((e for e in updated if e["_id"] == "product-p0002687"), None)
print(f"Updated products: {hit_up}")

# 3. Pobierz aktualny URL z Sanity
TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
H = {"Authorization": f"Bearer {TOKEN}"}
r = requests.get("https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production",
                 headers=H, params={"query": '*[_type=="product" && sku=="P-0002687"] { "img": images[0].asset->url }'})
img_url = r.json()["result"][0]["img"]
print(f"Aktualny URL Sanity: {img_url}")

# 4. Pobierz miniaturę i sprawdź czy jest czarny piksel
thumb = img_url + "?w=256&h=256&fit=max"
resp = requests.get(thumb, headers=H, timeout=15)
img = Image.open(io.BytesIO(resp.content)).convert("RGB")
arr = np.array(img)
black_pix = np.all(arr < 40, axis=2).sum()
total = arr.shape[0] * arr.shape[1]
print(f"Miniatura {arr.shape}: czarne piksele {black_pix}/{total} = {black_pix/total*100:.1f}%")
print(f"Min jasność: {arr.min()}, Max jasność: {arr.max()}")


Flood-fill log: {'_id': 'product-p0002687', 'url': 'https://cdn.sanity.io/images/nzcwegq7/production/7695fecbea4a989d35369abcd01edf0e0612f542-1200x1200.webp', 'status': 'updated(192KB)'}
Updated products: {'_id': 'product-p0002687', 'url': 'https://cdn.sanity.io/images/nzcwegq7/production/7695fecbea4a989d35369abcd01edf0e0612f542-1200x1200.webp', 'status': 'updated(192KB)'}
Aktualny URL Sanity: https://cdn.sanity.io/images/nzcwegq7/production/f77b98e2ecc0a473c9b60140b20aa1b22cd75765-1200x1200.jpg


Miniatura (256, 256, 3): czarne piksele 46962/65536 = 71.7%
Min jasność: 0, Max jasność: 252


In [7]:

import requests, json, io
from PIL import Image
import numpy as np

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
H = {"Authorization": f"Bearer {TOKEN}"}
UA = {"User-Agent": "Mozilla/5.0 Chrome/120 Safari/537.36"}

# Oryginalne zdjęcie bechcicki.pl dla P-0002687
# SKU digits: 0002687 → n=0002687
n = "0002687"
bech_url = f"https://static.www.bechcicki.pl/P-/{n[0:2]}/{n[2:4]}/{n[4:6]}/{n[6]}/1/BIG.webp"
print(f"Pobieranie oryginału: {bech_url}")

r = requests.get(bech_url, headers=UA, timeout=20)
print(f"Status: {r.status_code}, rozmiar: {len(r.content)} bytes")

if r.status_code == 200 and len(r.content) > 5000:
    # Podgląd oryginalnego zdjęcia
    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    arr = np.array(img)
    black = np.all(arr < 40, axis=2).sum()
    total = arr.shape[0] * arr.shape[1]
    print(f"Oryginał: {arr.shape}, czarne piksele: {black}/{total} = {black/total*100:.1f}%")
    
    # Upload do Sanity
    upload_url = f"https://{PROJECT}.api.sanity.io/v2021-06-07/assets/images/{DATASET}"
    resp = requests.post(upload_url, headers={**H,"Content-Type":"image/webp"},
                        params={"filename":"p-0002687_orig.webp"},
                        data=r.content, timeout=60)
    print(f"Upload status: {resp.status_code}")
    
    if resp.status_code in (200,201):
        asset_id = resp.json()["document"]["_id"]
        print(f"Asset ID: {asset_id}")
        
        # Patch produktu
        mutation = {"mutations":[{"patch":{"id":"product-p0002687","set":{
            "images":[{"_type":"image","asset":{"_type":"reference","_ref":asset_id}}]
        }}}]}
        patch_r = requests.post(f"https://{PROJECT}.api.sanity.io/v2021-06-07/data/mutate/{DATASET}",
                               headers={**H,"Content-Type":"application/json"},
                               json=mutation, timeout=30)
        print(f"Patch status: {patch_r.status_code}")
        if patch_r.status_code in (200,201):
            print("✅ Bloczek H+H przywrócony do oryginału bechcicki.pl!")
else:
    print("❌ Nie udało się pobrać oryginału")


Pobieranie oryginału: https://static.www.bechcicki.pl/P-/00/02/68/7/1/BIG.webp
Status: 200, rozmiar: 286236 bytes
Oryginał: (1200, 1200, 3), czarne piksele: 0/1440000 = 0.0%


Upload status: 200
Asset ID: image-7695fecbea4a989d35369abcd01edf0e0612f542-1200x1200-webp


Patch status: 200
✅ Bloczek H+H przywrócony do oryginału bechcicki.pl!


In [11]:

import requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
H = {"Authorization": f"Bearer {TOKEN}"}
r = requests.get(
    "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production",
    headers=H,
    params={"query": '*[_type=="product" && sku=="P-0002687"] { "img": images[0].asset->url, name }'}
)
result = r.json()["result"][0]
print(f"Produkt: {result['name']}")
print(f"Aktualny URL: {result['img']}")


Produkt: Bloczek H+H Gold+ 4,0-500 PWU 175/250/625 mm
Aktualny URL: https://cdn.sanity.io/images/nzcwegq7/production/7695fecbea4a989d35369abcd01edf0e0612f542-1200x1200.webp


In [4]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
URL = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
H = {"Authorization": f"Bearer {TOKEN}"}

def q(query):
    return requests.get(URL, headers=H, params={"query": query}).json()["result"]

total        = q('count(*[_type=="product"])')[0]
with_cat     = q('count(*[_type=="product" && defined(category)])')[0]
with_img     = q('count(*[_type=="product" && defined(images) && count(images)>0])')[0]
with_specs   = q('count(*[_type=="product" && defined(technicalSpec) && count(technicalSpec)>0])')[0]
with_slug    = q('count(*[_type=="product" && defined(slug)])')[0]
with_brand   = q('count(*[_type=="product" && defined(brand)])')[0]
categories   = q('count(*[_type=="category"])')[0]
top_cats     = q('count(*[_type=="category" && !defined(parent)])')[0]

print("=" * 52)
print("  WERYFIKACJA KOŃCOWA KATALOGU SANITY")
print("=" * 52)
print(f"  Produkty łącznie:          {total:>6}")
print(f"  Z kategorią:               {with_cat:>6}  ({with_cat/total*100:.1f}%)")
print(f"  Ze zdjęciem:               {with_img:>6}  ({with_img/total*100:.1f}%)")
print(f"  Bez zdjęcia:               {total-with_img:>6}  ({(total-with_img)/total*100:.1f}%)")
print(f"  Z parametrami technicz.:   {with_specs:>6}  ({with_specs/total*100:.1f}%)")
print(f"  Z slugiem:                 {with_slug:>6}  ({with_slug/total*100:.1f}%)")
print(f"  Z marką:                   {with_brand:>6}  ({with_brand/total*100:.1f}%)")
print(f"  Kategorie łącznie:         {categories:>6}")
print(f"  Kategorie top-level:       {top_cats:>6}")
print("=" * 52)
print(f"  Kompletność danych:  {'✅ BARDZO DOBRA' if with_cat/total>0.99 and with_img/total>0.9 else '⚠️ Wymaga uwagi'}")


TypeError: 'int' object is not subscriptable

In [8]:

import requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
URL = "https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production"
H   = {"Authorization": f"Bearer {TOKEN}"}

def q(query):
    return requests.get(URL, headers=H, params={"query": query}).json()["result"]

total      = q('count(*[_type=="product"])')
with_cat   = q('count(*[_type=="product" && defined(category)])')
with_img   = q('count(*[_type=="product" && defined(images) && count(images)>0])')
with_specs = q('count(*[_type=="product" && defined(technicalSpec) && count(technicalSpec)>0])')
with_slug  = q('count(*[_type=="product" && defined(slug)])')
with_brand = q('count(*[_type=="product" && defined(brand)])')
categories = q('count(*[_type=="category"])')
top_cats   = q('count(*[_type=="category" && !defined(parent)])')

print("=" * 52)
print("  WERYFIKACJA KOŃCOWA KATALOGU SANITY")
print("=" * 52)
print(f"  Produkty łącznie:          {total:>6}")
print(f"  Z kategorią:               {with_cat:>6}  ({with_cat/total*100:.1f}%)")
print(f"  Ze zdjęciem:               {with_img:>6}  ({with_img/total*100:.1f}%)")
print(f"  Bez zdjęcia:               {total-with_img:>6}  ({(total-with_img)/total*100:.1f}%)")
print(f"  Z parametrami tech.:       {with_specs:>6}  ({with_specs/total*100:.1f}%)")
print(f"  Ze slugiem:                {with_slug:>6}  ({with_slug/total*100:.1f}%)")
print(f"  Z marką:                   {with_brand:>6}  ({with_brand/total*100:.1f}%)")
print(f"  Kategorie łącznie:         {categories:>6}")
print(f"  Kategorie top-level:       {top_cats:>6}")
print("=" * 52)
ok = with_cat/total > 0.99 and with_img/total > 0.90
print(f"  Status: {'✅ BARDZO DOBRA' if ok else '⚠️ Wymaga uwagi'}")


  WERYFIKACJA KOŃCOWA KATALOGU SANITY
  Produkty łącznie:           15836
  Z kategorią:                15836  (100.0%)
  Ze zdjęciem:                14823  (93.6%)
  Bez zdjęcia:                 1013  (6.4%)
  Z parametrami tech.:        15836  (100.0%)
  Ze slugiem:                 15836  (100.0%)
  Z marką:                    15836  (100.0%)
  Kategorie łącznie:            551
  Kategorie top-level:           10
  Status: ✅ BARDZO DOBRA
